For casme dataset, load the data per person from ftp server, one after other in a thread in background, it has frames stored, get only the color values, no depth needed, store them to ./data/casme/person_id/, this will be fed to the motion amplification.

## CASME3 Data Loader (FTP)

This script connects to the CASME3 FTP server, searches for subject directories containing "color" folders, and downloads the frames to `./data/casme/{subject_id}/`. 
It runs in a background thread so you can continue working while data downloads.

**Instructions:**
1. Update `FTP_HOST`, `FTP_USER`, and `FTP_PASS` with your credentials.
2. Run the cell below.
3. The download will happen in the background. Check the `./data/casme` folder for progress.

In [1]:
import queue
import threading
import time
import ftplib
import os
import shutil
import zipfile
import pandas as pd
import re
from dotenv import load_dotenv
# Use tqdm.auto to fallback if widgets are missing/broken
from tqdm.auto import tqdm

STOP_SIGNAL = "STOP"
load_dotenv()


def on_rm_error(func, path, exc_info):
    """
    Error handler for shutil.rmtree.
    If the error is due to an access error (read only file)
    it attempts to add write permission and then retries.
    If the error is due to the file being used by another process,
    it waits a bit and retries.
    """
    import stat
    # Is the error an access error?
    if not os.access(path, os.W_OK):
        os.chmod(path, stat.S_IWRITE)
        try:
            func(path)
            return
        except Exception:
            pass
    
    # Maybe locked?
    print(f"Warning: Could not delete {path}. Retrying in 1s...")
    time.sleep(1)
    try:
        # Try to chmod again just in case
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception as e:
        print(f"Failed to force delete {path}: {e}")

def robust_rmtree(path, retries=5, delay=1.0):
    if not os.path.exists(path):
        return
    
    try:
        import stat
        os.chmod(path, stat.S_IWRITE)
    except:
        pass

    for i in range(retries):
        try:
            import gc
            gc.collect()
            
            shutil.rmtree(path, onerror=on_rm_error)
            print(f"Successfully deleted {path}")
            return
        except OSError as e:
            if i < retries - 1:
                time.sleep(delay)
            else:
                print(f"Warning: Failed to delete {path} after {retries} attempts: {e}")
                if os.name == 'nt':
                    try:
                        print("Attempting system shell delete...")
                        os.system(f'rmdir /S /Q "{path}"')
                    except Exception as sys_e:
                        print(f"System shell delete failed: {sys_e}")


def download_casme3_generator(host, user, password, local_root_dir, excel_path, parts_config, stop_event):
    """
    Iterates through dataset parts (A, B... per config), downloads annotated subject zips, 
    extracts them, and yields the local path.
    """
    print(f"Connecting to FTP {host}...")
    ftp = None
    try:
        ftp = ftplib.FTP(host)
        ftp.login(user, password)
    except Exception as e:
        print(f"FTP Connection Failed: {e}")
        return

    # 1. Load Annotations just for filtering subjects to download
    valid_subjects = set()
    try:
        if excel_path and os.path.exists(excel_path):
            try:
                # Load excel purely to find valid subjects
                df = pd.read_excel(excel_path)
                # Heuristic to find subject column
                valid_col = None
                for col in df.columns:
                    if 'sub' in str(col).lower():
                        valid_col = col
                        break
                
                if valid_col:
                    # robust ID extraction
                    raw_list = df[valid_col].astype(str).tolist()
                    for item in raw_list:
                        match = re.search(r'(\d+)', item)
                        if match:
                            num_str = match.group(1)
                            valid_subjects.add(num_str) 
                            valid_subjects.add(str(int(num_str)))
                    
                    print(f"Loaded {len(valid_subjects)} valid subject IDs from annotations.")
                else:
                    print(f"Warning: Could not find 'Subject' column in {excel_path}. Processing all.")
            except Exception as e:
                print(f"Error reading Excel for subject filtering: {e}")
                # Continue without filtering if excel read fails
        else:
            print(f"Annotation file not found at {excel_path}. Processing all subjects found on FTP.")
    except Exception as e:
        print(f"Error during init: {e}")

    try:
        # 2. Iterate Configured Parts
        for part_name, relative_path in parts_config.items():
            if stop_event.is_set(): break

            print(f"Searching {part_name} at {relative_path}...")
            
            file_list = []
            try:
                ftp.retrlines(f'NLST {relative_path}', file_list.append)
                print(f"Found {len(file_list)} files in {part_name}")
            except ftplib.error_perm as e:
                print(f"Skipping {part_name} (Not found or No Access): {e}")
                continue

            file_list.sort() 
            
            processed_count = 0
            
            # Use tqdm for progress bar on file list
            # position=0 ensures it stays at top, leave=True keeps it when done
            pbar = tqdm(file_list, desc=f"Scanning {part_name}", unit="file", position=0, leave=True)
            
            for zip_filename in pbar:
                if stop_event.is_set(): 
                    pbar.close()
                    print("Stop signal received. Aborting download loop.")
                    return
                
                # Check extension (case insensitive)
                if not zip_filename.lower().endswith('.zip'):
                    continue
                
                base_name = os.path.basename(zip_filename.replace('\\', '/'))
                match = re.search(r'(\d+)', base_name)
                
                if not match:
                    continue
                
                subject_id_raw = match.group(1)
                subject_int = str(int(subject_id_raw)) # "01" -> "1"
                
                # Filter Logic
                is_valid = (not valid_subjects) or \
                           (subject_id_raw in valid_subjects) or \
                           (subject_int in valid_subjects)

                if valid_subjects and not is_valid:
                    # pbar.set_description(f"Skipping {zip_filename}")
                    continue

                safe_id = f"sub{subject_id_raw}_{part_name}"
                extract_dir = os.path.join(local_root_dir, safe_id)
                local_zip_path = os.path.join(local_root_dir, f"{safe_id}.zip")

                # Check if cached
                if os.path.exists(extract_dir) and len(os.listdir(extract_dir)) > 0:
                    pbar.write(f"Found cached data for {safe_id}. Yielding...")
                    yield extract_dir
                    continue
                
                robust_rmtree(extract_dir)
                os.makedirs(extract_dir, exist_ok=True)

                pbar.set_description(f"Downloading {zip_filename}...")
                
                if '/' in zip_filename or '\\' in zip_filename:
                     remote_file = zip_filename.replace('\\', '/')
                else:
                     remote_file = f"{relative_path}/{zip_filename}"
                
                remote_file = remote_file.replace('//', '/')

                # Download ZIP with progress
                try:
                    file_size = 0
                    try:
                        file_size = ftp.size(remote_file)
                    except:
                        pass
                    
                    with open(local_zip_path, 'wb') as f:
                        if file_size > 0:
                            # Inner progress bar for download bytes
                            # position=1 try to put it under the main bar
                            with tqdm(total=file_size, unit='B', unit_scale=True, desc=f"DL {safe_id}", leave=False, position=1) as dl_pbar:
                                def callback(data):
                                    f.write(data)
                                    dl_pbar.update(len(data))
                                ftp.retrbinary(f"RETR {remote_file}", callback)
                        else:
                            ftp.retrbinary(f"RETR {remote_file}", f.write)
                    
                    if os.path.getsize(local_zip_path) < 100:
                        pbar.write(f"Warning: Downloaded file {local_zip_path} is too small. Deleting.")
                        try: os.remove(local_zip_path)
                        except: pass
                        continue

                except Exception as e:
                    pbar.write(f"Download failed for {zip_filename} (Path: {remote_file}): {e}")
                    try: os.remove(local_zip_path)
                    except: pass
                    robust_rmtree(extract_dir)
                    continue

                if stop_event.is_set():
                    try: os.remove(local_zip_path)
                    except: pass
                    return

                # Extract ZIP
                pbar.set_description(f"Extracting {safe_id}...")
                try:
                    with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
                        # Ensure we don't hold the file open implicitly
                        zip_ref.extractall(extract_dir)
                    
                    if len(os.listdir(extract_dir)) == 0:
                         pbar.write(f"Warning: {zip_filename} extraction resulted in empty dir.")
                    else:
                         yield extract_dir
                         processed_count += 1
                         
                except zipfile.BadZipFile:
                    pbar.write(f"Corrupt Zip File: {zip_filename}")
                except Exception as e:
                    pbar.write(f"Extraction Error: {e}")
                finally:
                    # Explicitly remove zip file (cleanup zip immediately to save space)
                    try:
                        if os.path.exists(local_zip_path):
                            os.remove(local_zip_path)
                    except Exception as e:
                        pbar.write(f"Warning: Could not remove zip {local_zip_path}: {e}")
            
            if processed_count == 0:
                print(f"Warning: No valid zip files found or downloaded in {part_name}")

    except Exception as e:
        print(f"Generator Error: {e}")
    finally:
        if ftp:
            try: ftp.quit() 
            except: pass


def _producer_thread(host, user, password, local_root, excel_path, config, q, stop_event):
    """
    Driven by the generator. Puts resulting paths into the provided queue.
    """
    try:
        gen = download_casme3_generator(host, user, password, local_root, excel_path, config, stop_event)
        for path in gen:
            if stop_event.is_set(): break
            while not stop_event.is_set():
                try:
                    q.put(path, timeout=1)
                    break 
                except queue.Full:
                    continue
    except Exception as e:
        print(f"Producer Thread crashed: {e}")
    finally:
        # Use put_nowait or timeout to prevent hanging if queue is full and consumer is gone
        try:
            q.put(STOP_SIGNAL, timeout=5)
        except queue.Full:
            print("Warning: Could not send STOP signal (Queue full).")
        print("Background download process finished.")


class CASMEDataLoader:
    def __init__(self, excel_path, local_root="./data/casme_raw", processed_root="./data/casme_processed", buffer_size=1):
        # Load from .env
        self.host = os.getenv("FTP_HOST")
        self.user = os.getenv("FTP_USER")
        self.password = os.getenv("FTP_PASS")
        
        if not all([self.host, self.user, self.password]):
            raise ValueError("Missing FTP credentials in .env file (FTP_HOST, FTP_USER, FTP_PASS)")
            
        self.excel_path = excel_path
        self.local_root = local_root
        # Output directory for Onset/Apex frames
        self.processed_root = processed_root
        
        # Configure FTP paths per dataset structure (update if needed)
        self.parts_config = {
            "Part_A": "part_A/data/Compressed_version1_seperate_compress",
            "Part_B": "part_B/Compressed_version1_seperate_compress", 
        }
        
        # Queue loads "raw" subject paths
        self.data_queue = queue.Queue(maxsize=buffer_size) 
        self._thread = None
        self._stop_event = threading.Event()

        # Load annotations for onset/apex extraction
        self.annotations = {} # sub_id -> list of {video, onset, apex, emotion}
        self._load_annotations()
        
    def _load_annotations(self):
        if not os.path.exists(self.excel_path):
            print(f"Error: Annotation file not found at {self.excel_path}")
            return

        try:
            df = pd.read_excel(self.excel_path)
            # Find required columns heuristically
            col_map = {c.lower(): c for c in df.columns}
            
            # Subject
            sub_col = col_map.get('subject') or col_map.get('sub')
            if not sub_col:
                 matches = [c for c in col_map.values() if 'sub' in c.lower()]
                 if matches: sub_col = matches[0]

            # Sequence/Video (prioritize 'seq', then 'video', then 'file')
            vid_col = col_map.get('seq') or col_map.get('filename') or col_map.get('video')
            if not vid_col:
                 matches = [c for c in col_map.values() if 'seq' in c.lower() or 'file' in c.lower() or 'video' in c.lower()]
                 if matches: vid_col = matches[0]

            # Onset / Apex
            onset_col = col_map.get('onset') or [c for c in col_map.values() if 'onset' in c.lower()][0]
            apex_col = col_map.get('apex') or [c for c in col_map.values() if 'apex' in c.lower()][0]

            # Emotion (optional)
            emo_col = col_map.get('emotion') or col_map.get('est_emotion') or col_map.get('label')
            if not emo_col:
                 matches = [c for c in col_map.values() if 'emo' in c.lower() or 'label' in c.lower()]
                 if matches: emo_col = matches[0]

            print(f"Using columns: Subject='{sub_col}', Video='{vid_col}', Onset='{onset_col}', Apex='{apex_col}', Emotion='{emo_col}'")
            
            count_loaded = 0
            for idx, row in df.iterrows():
                try:
                    # Extract subject ID
                    sub_raw = str(row[sub_col])
                    match = re.search(r'(\d+)', sub_raw)
                    if match:
                        sub_id = str(int(match.group(1))) # Normalize '01' to '1'
                        
                        if sub_id not in self.annotations:
                            self.annotations[sub_id] = []
                        
                        if pd.notna(row[onset_col]) and pd.notna(row[apex_col]):
                            record = {
                                'video': str(row[vid_col]).strip(),
                                'onset': int(row[onset_col]),
                                'apex': int(row[apex_col])
                            }
                            if emo_col and pd.notna(row[emo_col]):
                                record['emotion'] = str(row[emo_col]).strip()
                                
                            self.annotations[sub_id].append(record)
                            count_loaded += 1
                except Exception as e:
                    continue 
            
            print(f"Loaded annotations for {len(self.annotations)} subjects ({count_loaded} videos).")

        except Exception as e:
            print(f"Failed to load annotations: {e}")

    def start(self):
        """Starts the background downloading thread."""
        if self._thread and self._thread.is_alive():
            print("Loader already running.")
            return
            
        self._stop_event.clear()
        self._thread = threading.Thread(
            target=_producer_thread,
            args=(self.host, self.user, self.password, self.local_root, self.excel_path, self.parts_config, self.data_queue, self._stop_event)
        )
        self._thread.daemon = True
        self._thread.start()
        print("Background data loading started...")

    def stop(self):
        """sends stop signal to background thread."""
        print("Stopping background loader...")
        self._stop_event.set()
        
        # Drain the queue
        try:
            while not self.data_queue.empty():
                self.data_queue.get_nowait()
        except:
            pass

    def get_next_subject(self):
        """
        Returns (subject_path) for next subject.
        Blocks if waiting for download.
        Returns None if exhausted.
        """
        while True:
            try:
                item = self.data_queue.get(block=True, timeout=1) 
                
                if item == STOP_SIGNAL:
                    self.data_queue.put(STOP_SIGNAL) 
                    return None
                    
                return item
                
            except queue.Empty:
                if self._stop_event.is_set():
                    return None
                    
                if self._thread and not self._thread.is_alive():
                    return None
                continue 
    
    def process_and_save(self, subject_path):
        """
        Extracts onset and apex frames for all videos of this subject 
        and saves them to processed_root.
        Returns the path to the processed subject folder.
        """
        # Extract subject ID from path (e.g. .../sub01_Part_A -> "1")
        base = os.path.basename(subject_path)
        match = re.search(r'sub(\d+)', base)
        if not match:
            print(f"Could not parse subject ID from {base}")
            return None
        
        sub_id = str(int(match.group(1)))
        
        if sub_id not in self.annotations:
            print(f"No annotations found for subject {sub_id} (Path: {base})")
            return None

        # Build output path in "clean" directory
        out_sub_dir = os.path.join(self.processed_root, f"sub{sub_id}")
        os.makedirs(out_sub_dir, exist_ok=True)
        
        processed_videos = 0

        # Debug: list first few dirs in subject path
        print(f"Debug: Listing dirs in {subject_path}:")
        try:
             print([d for d in os.listdir(subject_path) if os.path.isdir(os.path.join(subject_path, d))][:5])
        except:
             print("Could not list dirs")

        # Iterate through annotated videos for this subject
        for video_info in self.annotations[sub_id]:
            video_info_vid = video_info['video'] # video_info['video'] is e.g. "aa"
            onset_frame = video_info['onset']
            apex_frame = video_info['apex']
            
            # Find the video folder in the extracted path
            # Search logic: exact match OR nested match OR subXX_video match
            video_source_path = None
            
            # 1. Direct path check (common case)
            direct_try = os.path.join(subject_path, video_info_vid)
            if os.path.exists(direct_try):
                video_source_path = direct_try
            
            # 2. Recursive search with robust checks
            if not video_source_path:
                v_low = video_info_vid.lower()
                for root, dirs, files in os.walk(subject_path):
                    for d in dirs:
                        d_low = d.lower()
                        # Checks:
                        # 1. Exact match 'a' == 'a'
                        # 2. Sub-prefixed 'sub01_a' matches 'a'
                        if d_low == v_low:
                            video_source_path = os.path.join(root, d)
                            break
                        
                        # Robust suffix check: ends with _video (e.g. sub1_a)
                        # Avoid 'Part_A' matching 'a' via simple endswith
                        # We demand 'subXX_video' pattern or similar
                        if re.search(rf"sub\d+_{re.escape(v_low)}$", d_low):
                            video_source_path = os.path.join(root, d)
                            break
                            
                    if video_source_path: break
            
            if not video_source_path:
                print(f"Video '{video_info_vid}' folder not found in {subject_path}")
                continue

            # Find specific frames
            try:
                files = sorted(os.listdir(video_source_path))
            except Exception:
                continue
            
            def get_frame_file(target_num):
                # Optimization: check known formats first
                candidates = [f"img{target_num}.jpg", f"reg_img{target_num}.jpg", f"{target_num}.jpg"]
                for c in candidates:
                    if c in files:
                        return os.path.join(video_source_path, c)

                # Use regex to find number in filenames
                for f in files:
                    if not f.lower().endswith(('.jpg', '.png', '.bmp')): continue
                    # Extract ALL numbers
                    nums = re.findall(r'\d+', f)
                    for n in nums:
                        if int(n) == target_num:
                            return os.path.join(video_source_path, f)
                return None

            onset_path = get_frame_file(onset_frame)
            apex_path = get_frame_file(apex_frame)
            
            if onset_path and apex_path:
                # Unique folder for this clip
                # Use video name AND onset/apex frames to ensure uniqueness
                # e.g. "a_37_63"
                clip_dirname = f"{video_info_vid}_{onset_frame}_{apex_frame}"
                out_vid_dir = os.path.join(out_sub_dir, clip_dirname)
                
                os.makedirs(out_vid_dir, exist_ok=True)
                
                shutil.copy2(onset_path, os.path.join(out_vid_dir, "onset.jpg"))
                shutil.copy2(apex_path, os.path.join(out_vid_dir, "apex.jpg"))
                
                # Write metadata
                with open(os.path.join(out_vid_dir, "info.txt"), "w") as f:
                    f.write(f"Subject: {sub_id}\nVideo: {video_info_vid}\nOnset: {onset_frame}\nApex: {apex_frame}\n")
                    if 'emotion' in video_info:
                        f.write(f"Emotion: {video_info['emotion']}\n")
                
                processed_videos += 1

        print(f"Processed {processed_videos} videos for subject {sub_id}")
        return out_sub_dir

    def cleanup_subject(self, subject_path):
        """
        Deletes the local raw data for a subject with retry logic.
        """
        robust_rmtree(subject_path)


In [ ]:
# Usage Example
excel_file = r"data\casme\cas(me)3_part_A_MaE_label_JpgIndex_v2_emotion.xlsx"

# Initialized from .env
# Start the loader which downloads subjects in background
loader = CASMEDataLoader(excel_file, local_root="./data/casme_raw", processed_root="./data/casme_processed") 

try:
    loader.start()
    
    # Process subjects as they arrive in the queue
    subjects_processed_count = 0
    
    while True:
        print("\nWaiting for next subject download...")
        # Get path to raw downloaded subject folder
        subject_raw_path = loader.get_next_subject()
        
        if subject_raw_path is None:
            print("No more subjects to download/process.")
            break
            
        print(f"Raw data ready at: {subject_raw_path}")
        
        # PROCESS: Extract Onset/Apex frames based on Excel
        processed_path = loader.process_and_save(subject_raw_path)
        
        if processed_path:
            print(f"Saved processed frames to: {processed_path}")
        else:
            print(f"No valid videos processed for {subject_raw_path}")

        # CLEANUP: Delete the raw download to save space
        print(f"Cleaning up raw files for {subject_raw_path}...")
        loader.cleanup_subject(subject_raw_path)
        
        subjects_processed_count += 1
        
        # For testing, break after a few. Remove this break to run fully.
        if subjects_processed_count >= 2:
            print("Test run complete (processed 2 subjects). Stopping loader loop.")
            break

except KeyboardInterrupt:
    print("\nUser interrupted processing.")
finally:
    loader.stop()
    print("All downloads finished or stopped.")


Using columns: Subject='sub', Video='seq', Onset='onset', Apex='apex', Emotion='emotion'
Loaded annotations for 98 subjects (3346 videos).
Connecting to FTP 74.220.215.205...
Background data loading started...

Waiting for next subject download...
Loaded 98 valid subject IDs from annotations.
Searching Part_A at part_A/data/Compressed_version1_seperate_compress...
Found 102 files in Part_A


Scanning Part_A:   0%|          | 0/102 [00:00<?, ?file/s]

Successfully deleted ./data/casme_raw\sub1_Part_A


DL sub1_Part_A:   0%|          | 0.00/3.82G [00:00<?, ?B/s]